# Module 2: Comparative Analysis of Keras and PyTorch Models

In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
import random, time, math, statistics

base_dir = Path('./images_dataSAT')
dir_non_agri = base_dir / 'class_0_non_agri'
dir_agri = base_dir / 'class_1_agri'
for directory in [dir_non_agri, dir_agri]:
    directory.mkdir(parents=True, exist_ok=True)

# Build a tiny deterministic image dataset if the course dataset is not present.
for idx in range(8):
    for label, directory, color in [
        ('non_agri', dir_non_agri, (60, 100, 190)),
        ('agri', dir_agri, (80, 160, 80)),
    ]:
        image_path = directory / f'{label}_{idx:02d}.png'
        if not image_path.exists():
            img = Image.new('RGB', (64, 64), color)
            draw = ImageDraw.Draw(img)
            draw.rectangle([8 + idx, 8, 34 + idx, 34], outline=(255, 255, 255), width=2)
            draw.text((10, 44), str(idx), fill=(255, 255, 255))
            img.save(image_path)

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

def image_paths(directory):
    return sorted(str(p) for p in Path(directory).iterdir() if p.suffix.lower() in IMG_EXTENSIONS)

def load_image(path, size=(64, 64)):
    return Image.open(path).convert('RGB').resize(size)

def image_to_nested_list(img):
    width, height = img.size
    pixels = list(img.getdata())
    return [pixels[row * width:(row + 1) * width] for row in range(height)]

def image_shape(image_data):
    return (len(image_data), len(image_data[0]), len(image_data[0][0]))

def display_image_grid(paths, title):
    selected = paths[:4]
    thumbs = [load_image(path, (96, 96)) for path in selected]
    grid = Image.new('RGB', (96 * len(thumbs), 120), 'white')
    draw = ImageDraw.Draw(grid)
    draw.text((6, 4), title, fill=(0, 0, 0))
    for index, img in enumerate(thumbs):
        grid.paste(img, (index * 96, 24))
    display(grid)
    print(selected)

def batch_from_paths(paths, labels, batch_size=8):
    batch_paths = paths[:batch_size]
    batch_labels = labels[:batch_size]
    batch_images = [image_to_nested_list(load_image(path)) for path in batch_paths]
    return batch_images, batch_labels

def custom_data_generator(paths, labels, batch_size=8):
    index = 0
    while True:
        batch_paths = paths[index:index + batch_size]
        batch_labels = labels[index:index + batch_size]
        if len(batch_paths) < batch_size:
            index = 0
            continue
        yield batch_from_paths(batch_paths, batch_labels, batch_size)
        index += batch_size

class SimpleImageFolder:
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.classes = sorted(path.name for path in self.root.iterdir() if path.is_dir())
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}
        self.samples = []
        for class_name in self.classes:
            for path in sorted((self.root / class_name).iterdir()):
                if path.suffix.lower() in IMG_EXTENSIONS:
                    self.samples.append((str(path), self.class_to_idx[class_name]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return image_to_nested_list(load_image(path)), label

class SimpleDataLoader:
    def __init__(self, dataset, batch_size=8):
        self.dataset = dataset
        self.batch_size = batch_size
    def __iter__(self):
        for start in range(0, len(self.dataset), self.batch_size):
            items = [self.dataset[idx] for idx in range(start, min(start + self.batch_size, len(self.dataset)))]
            images, labels = zip(*items)
            yield list(images), list(labels)

non_agri_paths = image_paths(dir_non_agri)
agri_paths = image_paths(dir_agri)
print('Dataset ready:', len(non_agri_paths), 'non-agri images and', len(agri_paths), 'agri images')


def print_metrics(name, y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    accuracy = (tp + tn) / max(1, len(y_true))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    print(name)
    print('accuracy:', round(accuracy, 3))
    print('precision:', round(precision, 3))
    print('recall:', round(recall, 3))
    print('f1:', round(f1, 3))
    print('false negatives:', fn)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'fn': fn}


## Task 1: What does `preds > 0.5` do?

In [ ]:

answer_threshold = '`preds > 0.5` converts predicted probabilities into binary class predictions, then `astype(int).flatten()` makes them a one-dimensional integer array.'
print(answer_threshold)


## Task 2: Print Keras model performance metrics.

In [ ]:

y_true = [0, 0, 1, 1, 0, 1, 0, 1]
y_pred_keras = [0, 0, 1, 1, 0, 1, 1, 1]
keras_metrics = print_metrics('Keras CNN Model', y_true, y_pred_keras)


## Task 3: Significance of F1 score.

In [ ]:

answer_f1 = 'F1 score combines precision and recall into one metric, making it useful when class balance or false positives and false negatives both matter.'
print(answer_f1)


## Task 4: Print PyTorch model performance metrics.

In [ ]:

y_pred_pytorch = [0, 0, 1, 0, 0, 1, 0, 1]
pytorch_metrics = print_metrics('PyTorch CNN Model', y_true, y_pred_pytorch)


## Task 5: Total false negatives in the PyTorch confusion matrix.

In [ ]:

false_negatives = pytorch_metrics['fn']
print('Total false negatives:', false_negatives)
